In [1]:
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.14.0+cu126
CUDA: True
GPU: NVIDIA GeForce GTX 1050 Ti
VRAM: 4.0 GB


In [2]:
model = YOLO("yolo11n.pt")

In [3]:
print(model)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=

In [4]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

DETECTION_CONFIG = (
    PROJECT_DIR / "configs" / "aerovision_detection.yaml"
)

print(DETECTION_CONFIG)
print(DETECTION_CONFIG.exists())

c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_detection.yaml
True


In [6]:
DATASET_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "drone_traffic_detection"
)

for split in ["train", "val", "test"]:
    images_dir = DATASET_DIR / "images" / split
    labels_dir = DATASET_DIR / "labels" / split

    image_count = len(list(images_dir.glob("*")))
    label_count = len(list(labels_dir.glob("*.txt")))

    print(
        f"{split}: "
        f"{image_count} imagens | "
        f"{label_count} labels"
    )

train: 950 imagens | 950 labels
val: 251 imagens | 251 labels
test: 118 imagens | 118 labels


In [7]:
diagnostic_results = model.train(
    data=str(DETECTION_CONFIG),
    epochs=1,
    imgsz=416,
    batch=2,
    workers=2,
    cache=False,
    device=0,
    project=str(PROJECT_DIR / "results" / "detection"),
    name="gpu_diagnostic",
)

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosai

In [8]:
batch_test_results = model.train(
    data=str(DETECTION_CONFIG),
    epochs=1,
    imgsz=416,
    batch=4,
    workers=2,
    cache=False,
    device=0,
    project=str(PROJECT_DIR / "results" / "detection"),
    name="batch4_test",
)

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\Andrezinho\OneDrive\Docu

In [9]:
baseline_results = model.train(
    data=str(DETECTION_CONFIG),
    epochs=30,
    imgsz=416,
    batch=4,
    workers=2,
    cache=False,
    device=0,
    project=str(PROJECT_DIR / "results" / "detection"),
    name="baseline",
)

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\configs\aerovision_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\Andrezinho\OneDrive\Doc

In [12]:
from pathlib import Path

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
    / "detection"
)

BEST_MODEL = RESULTS_DIR / "baseline-2" / "weights" / "best.pt"

print("Melhor modelo:")
print(BEST_MODEL)
print("Existe:", BEST_MODEL.exists())

Melhor modelo:
c:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\results\detection\baseline-2\weights\best.pt
Existe: True


In [13]:
best_model = YOLO(str(BEST_MODEL))

print("Modelo baseline carregado com sucesso.")

Modelo baseline carregado com sucesso.


In [14]:
test_results = best_model.val(
    data=str(DETECTION_CONFIG),
    split="test",
    imgsz=416,
    batch=4,
    device=0,
    project=str(PROJECT_DIR / "results" / "detection"),
    name="evaluation"
)

print("Avaliação concluída.")

Ultralytics 8.4.155  Python-3.11.0 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.20.2 ms, read: 114.834.3 MB/s, size: 26.4 KB)
val: Scanning C:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\data\processed\drone_traffic_detection\labels\test... 118 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 248.0it/s 0.5s1s
val: New cache created: C:\Users\Andrezinho\OneDrive\Documentos\GitHub\AeroVision\data\processed\drone_traffic_detection\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 14.7it/s 2.0s0.1s
                   all        118       1697      0.927      0.914      0.947       0.76
               bicycle         65        101      0.794      0.762      0.832      0.365
                   bus         82        116      0.966      0.985      0.995     

In [15]:
EVALUATION_DIR = (
    PROJECT_DIR
    / "results"
    / "detection"
    / "evaluation"
)

print("Arquivos da avaliação:\n")

for file in sorted(EVALUATION_DIR.iterdir()):
    print(file.name)

Arquivos da avaliação:

BoxF1_curve.png
BoxP_curve.png
BoxPR_curve.png
BoxR_curve.png
confusion_matrix.png
confusion_matrix_normalized.png
val_batch0_labels.jpg
val_batch0_pred.jpg
val_batch1_labels.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
val_batch2_pred.jpg
